# Residential Appliance Energy Consumption Prediction

## Data Understanding and Exploratory Data Analysis

## Project Objective

The objective of this project is to develop and evaluate machine learning
models for predicting residential appliance energy consumption using indoor
environmental measurements, outdoor weather conditions, and temporal
information.

The analysis will investigate the factors associated with appliance energy
demand, evaluate the predictive performance of linear and regularized
regression models, and later compare them with nonlinear machine learning
approaches.

Particular attention will be given to model generalization, data leakage,
feature selection, error analysis, and the operational interpretation of
predictions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

> **Reproducibility:** Download `energydata_complete.csv` from the UCI Appliances Energy Prediction dataset and save it as `data/energydata_complete.csv` before running this notebook.


In [ ]:
from pathlib import Path

data_path = Path("data/energydata_complete.csv")
if not data_path.exists():
    raise FileNotFoundError(
        "Download energydata_complete.csv from the UCI Appliances Energy Prediction dataset "
        "and place it in the data/ folder."
    )
df = pd.read_csv(data_path)


In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df.info()

In [ ]:
df["Appliances"]

In [ ]:
df["Appliances"].head()

In [ ]:
df["Appliances"].describe()

In [ ]:
df.isnull().sum()

In [ ]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percent": df.isnull().mean() * 100
})

missing_summary = missing_summary.sort_values(
    "Missing_Percent",
    ascending=False
)

missing_summary

In [ ]:
df.duplicated().sum()

In [ ]:
df["date"].head()

In [ ]:
df["date"] = pd.to_datetime(df["date"])

In [ ]:
numeric_df = df.select_dtypes(
    include=np.number
)

np.isinf(numeric_df).sum().sort_values(
    ascending=False
)

In [ ]:
# 1. Dataset dimensions
print("Dataset shape:", df.shape)

# 2. Missing values
print("Total missing values:", df.isnull().sum().sum())

# 3. Duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# 4. Data types
print(df.dtypes)

# 5. Convert timestamp
df["date"] = pd.to_datetime(df["date"])

# Timestamp integrity
print("Start date:", df["date"].min())
print("End date:", df["date"].max())
print("Unique timestamps:", df["date"].nunique())
print("Duplicate timestamps:", df["date"].duplicated().sum())

# 6. Sampling interval
print(
    df["date"]
      .sort_values()
      .diff()
      .value_counts()
      .head()
)

# 7. Numerical summary
display(df.describe().T)

# 8. Constant variables
constant_columns = [
    col for col in df.columns
    if df[col].nunique() == 1
]

print("Constant columns:", constant_columns)

# 9. Random/noise variables
display(
    df[["rv1", "rv2", "Appliances"]].corr()
)

In [ ]:
print(
    "rv1 and rv2 identical:",
    (df["rv1"] == df["rv2"]).all()
)

print(
    "Maximum absolute difference:",
    (df["rv1"] - df["rv2"]).abs().max()
)

## Exploratory Data Analysis

### Distribution of Appliance Energy Consumption

Understanding the distribution of the target variable is important for
identifying typical consumption levels, high-demand periods, potential
skewness, and unusual observations that may influence model performance.

In [ ]:
plt.figure(figsize=(10, 6))

plt.hist(
    df["Appliances"],
    bins=50,
    edgecolor="black"
)

plt.xlabel("Appliance Energy Consumption (Wh)")
plt.ylabel("Frequency")
plt.title("Distribution of Appliance Energy Consumption")

plt.show()

In [ ]:
target_skewness = df["Appliances"].skew()

print(
    "Target skewness:",
    round(target_skewness, 3)
)

In [ ]:
plt.figure(figsize=(10, 4))

plt.boxplot(
    df["Appliances"],
    vert=False
)

plt.xlabel("Appliance Energy Consumption (Wh)")
plt.title("Boxplot of Appliance Energy Consumption")

plt.show()

In [ ]:
Q1 = df["Appliances"].quantile(0.25)
Q3 = df["Appliances"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

In [ ]:
target_outliers = df[
    (df["Appliances"] < lower_bound) |
    (df["Appliances"] > upper_bound)
]

print(
    "Number of IQR-flagged observations:",
    len(target_outliers)
)

print(
    "Percentage:",
    round(
        len(target_outliers) / len(df) * 100,
        2
    ),
    "%"
)

In [ ]:
df.nlargest(
    10,
    "Appliances"
)[
    ["date", "Appliances", "lights"]
]

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    df["date"],
    df["Appliances"]
)

plt.xlabel("Date")
plt.ylabel("Appliance Energy Consumption (Wh)")
plt.title("Appliance Energy Consumption Over Time")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
daily_energy = (
    df.set_index("date")["Appliances"]
      .resample("D")
      .mean()
)

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    daily_energy.index,
    daily_energy.values
)

plt.xlabel("Date")
plt.ylabel("Mean Appliance Energy Consumption (Wh)")
plt.title("Daily Mean Appliance Energy Consumption")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
df["hour"] = df["date"].dt.hour
df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month

In [ ]:
df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

In [ ]:
hourly_energy = (
    df.groupby("hour")["Appliances"]
      .mean()
)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    hourly_energy.index,
    hourly_energy.values,
    marker="o"
)

plt.xlabel("Hour of Day")
plt.ylabel("Mean Appliance Energy Consumption (Wh)")
plt.title("Average Appliance Energy Consumption by Hour")

plt.xticks(range(0, 24))

plt.grid(alpha=0.3)

plt.show()

In [ ]:
weekday_energy = (
    df.groupby("day_of_week")["Appliances"]
      .mean()
)

In [ ]:
day_labels = [
    "Mon",
    "Tue",
    "Wed",
    "Thu",
    "Fri",
    "Sat",
    "Sun"
]

plt.figure(figsize=(10, 6))

plt.bar(
    day_labels,
    weekday_energy.values
)

plt.xlabel("Day of Week")
plt.ylabel("Mean Appliance Energy Consumption (Wh)")
plt.title("Average Appliance Energy Consumption by Day")

plt.show()

In [ ]:
df.groupby(
    "is_weekend"
)["Appliances"].agg(
    ["mean", "median", "std"]
)

### Key Findings: Target and Temporal Patterns

Appliance energy consumption is strongly right-skewed (skewness = 3.386),
with most observations concentrated at relatively low consumption levels and
a smaller number of high-demand periods.

Using the 1.5 × IQR criterion, 2,138 observations (10.83%) were flagged as
potential outliers. These observations were not automatically removed because
high energy-demand periods may represent legitimate household activity rather
than measurement error.

A clear time-of-day pattern was observed. Consumption was lowest during the
overnight and early-morning hours, increased during the morning, and reached
its highest average level around 18:00. Several of the largest individual
consumption observations also occurred during the late-afternoon and evening
period.

Mean consumption also varied across days of the week, with Monday showing the
highest average consumption and Tuesday the lowest within the observed study
period.

These findings suggest that temporal information contains potentially useful
predictive signal and should be considered during feature engineering.

In [ ]:
numeric_df = df.select_dtypes(include=np.number)

target_corr = (
    numeric_df
    .corr()["Appliances"]
    .sort_values(ascending=False)
)

target_corr

In [ ]:
target_corr = (
    numeric_df
    .corr()["Appliances"]
    .drop("Appliances")
    .sort_values()
)

In [ ]:
plt.figure(figsize=(10, 8))

target_corr.plot(kind="barh")

plt.xlabel("Pearson Correlation with Appliances")
plt.ylabel("Feature")
plt.title("Feature Correlation with Appliance Energy Consumption")

plt.tight_layout()
plt.show()

In [ ]:
corr_matrix = numeric_df.corr()

In [ ]:
upper_triangle = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape),
        k=1
    ).astype(bool)
)

high_corr_pairs = (
    upper_triangle
    .stack()
    .sort_values(
        key=abs,
        ascending=False
    )
)

high_corr_pairs.head(20)

### Predictor Correlation and Multicollinearity

The predictor correlation analysis revealed substantial multicollinearity
among the environmental sensor variables. The strongest relationship was
between the two random variables (`rv1` and `rv2`), which were perfectly
correlated.

Strong correlations were also observed among temperature measurements.
For example, `T6` and outdoor temperature (`T_out`) had a correlation of
approximately 0.975, while several indoor temperature sensors exhibited
correlations above 0.85.

Similar relationships were observed among relative-humidity sensors,
indicating that multicollinearity is a structural characteristic of the
sensor data rather than an isolated feature pair.

These findings provide motivation for evaluating regularized linear models.
Ridge regression may improve coefficient stability in the presence of
correlated predictors, while Lasso and Elastic Net will allow investigation
of whether a smaller subset of predictors can retain useful predictive
information.

A strong relationship was also observed between `T9` and month, which may
reflect seasonal temperature changes across the January-to-May observation
period.

In [ ]:
target_corr = (
    numeric_df
    .corr()["Appliances"]
    .drop("Appliances")
    .sort_values(ascending=False)
)

target_corr

In [ ]:
plt.figure(figsize=(10, 8))

target_corr.sort_values().plot(kind="barh")

plt.xlabel("Pearson Correlation with Appliances")
plt.ylabel("Feature")
plt.title("Feature Correlation with Appliance Energy Consumption")

plt.tight_layout()
plt.show()

In [ ]:
print("Strongest positive correlations:")
print(target_corr.head(10))

print("\nStrongest negative correlations:")
print(target_corr.tail(10))

In [ ]:
# Basic temporal features
df["hour"] = df["date"].dt.hour
df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# Cyclical hour encoding
df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)

# Cyclical day-of-week encoding
df["dow_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7
)

df["dow_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7
)

In [ ]:
df[
    [
        "date",
        "hour",
        "hour_sin",
        "hour_cos",
        "day_of_week",
        "dow_sin",
        "dow_cos",
        "month",
        "is_weekend"
    ]
].head(10)

### Temporal Feature Engineering

Exploratory analysis revealed substantial variation in appliance energy
consumption across hours of the day and days of the week. Temporal features
were therefore derived from the timestamp.

Hour and day-of-week are cyclical variables. Rather than representing them
only as integers, sine and cosine transformations were created so that
adjacent periods across cycle boundaries (for example, 23:00 and 00:00)
remain close in the feature space.

A weekend indicator and month variable were also retained. Because the
dataset covers only January through May, month is interpreted cautiously
and is not assumed to represent a complete annual seasonal cycle.

The raw timestamp will not be supplied directly to the regression models.

In [ ]:
df["date"].is_monotonic_increasing

In [ ]:
drop_columns = [
    "date",
    "Appliances",
    "hour",
    "day_of_week"
]

In [ ]:
X = df.drop(columns=drop_columns)

y = df["Appliances"]

In [ ]:
X.shape

In [ ]:
X.columns.tolist()

In [ ]:
n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

In [ ]:
X_train = X.iloc[:train_end]
y_train = y.iloc[:train_end]

X_val = X.iloc[train_end:val_end]
y_val = y.iloc[train_end:val_end]

X_test = X.iloc[val_end:]
y_test = y.iloc[val_end:]

In [ ]:
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

In [ ]:
print(
    "Train:",
    df.loc[:train_end - 1, "date"].min(),
    "to",
    df.loc[:train_end - 1, "date"].max()
)

print(
    "Validation:",
    df.loc[train_end:val_end - 1, "date"].min(),
    "to",
    df.loc[train_end:val_end - 1, "date"].max()
)

print(
    "Test:",
    df.loc[val_end:, "date"].min(),
    "to",
    df.loc[val_end:, "date"].max()
)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

## Validation Strategy and Leakage Prevention

Because the dataset consists of observations collected sequentially at
10-minute intervals, temporal ordering was preserved during model
evaluation. Rather than randomly distributing observations across training
and test sets, the data were divided chronologically into training,
validation, and final test periods.

This design provides a more conservative assessment of model generalization
to later observations and prevents future observations from directly
influencing model fitting.

Preprocessing transformations will be fitted using training data only and
subsequently applied to validation and test observations. Time-aware
cross-validation will also be used during hyperparameter tuning.

The initial modeling task is treated as concurrent energy estimation using
sensor and environmental measurements observed at the prediction timestamp.
A true future-forecasting system would require additional feature-availability
constraints, such as lagged sensor measurements or weather forecasts, and is
considered a potential extension.

In [ ]:
train_mean = y_train.mean()

print("Training target mean:", train_mean)

In [ ]:
y_val_baseline = np.full(
    len(y_val),
    train_mean
)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
baseline_mae = mean_absolute_error(
    y_val,
    y_val_baseline
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        y_val_baseline
    )
)

baseline_r2 = r2_score(
    y_val,
    y_val_baseline
)

In [ ]:
print(f"Baseline MAE:  {baseline_mae:.2f} Wh")
print(f"Baseline RMSE: {baseline_rmse:.2f} Wh")
print(f"Baseline R²:   {baseline_r2:.3f}")

In [ ]:
target_split_summary = pd.DataFrame({
    "Train": y_train.describe(),
    "Validation": y_val.describe(),
    "Test": y_test.describe()
})

target_split_summary

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
linear_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

In [ ]:
linear_pipeline.fit(
    X_train,
    y_train
)

In [ ]:
y_val_pred_linear = linear_pipeline.predict(
    X_val
)

In [ ]:
linear_mae = mean_absolute_error(
    y_val,
    y_val_pred_linear
)

linear_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        y_val_pred_linear
    )
)

linear_r2 = r2_score(
    y_val,
    y_val_pred_linear
)

In [ ]:
print(f"Linear Regression MAE:  {linear_mae:.2f} Wh")
print(f"Linear Regression RMSE: {linear_rmse:.2f} Wh")
print(f"Linear Regression R²:   {linear_r2:.3f}")

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Naive Mean Baseline",
        "Linear Regression"
    ],
    "MAE": [
        baseline_mae,
        linear_mae
    ],
    "RMSE": [
        baseline_rmse,
        linear_rmse
    ],
    "R2": [
        baseline_r2,
        linear_r2
    ]
})

results

In [ ]:
rmse_improvement = (
    (baseline_rmse - linear_rmse)
    / baseline_rmse
    * 100
)

print(
    f"RMSE improvement over baseline: "
    f"{rmse_improvement:.2f}%"
)

In [ ]:
mae_improvement = (
    (baseline_mae - linear_mae)
    / baseline_mae
    * 100
)

print(
    f"MAE improvement over baseline: "
    f"{mae_improvement:.2f}%"
)

### Baseline and Linear Regression Performance

The naive mean baseline produced an MAE of 53.97 Wh and an RMSE of
92.39 Wh, with an R² close to zero (-0.003), indicating that a constant
mean prediction provides essentially no explanatory value for the
validation-period energy consumption.

Linear Regression improved on the baseline, reducing MAE by approximately
5.1% and RMSE by approximately 6.8%. The model achieved an R² of 0.128,
indicating that approximately 12.8% of the variation in appliance energy
consumption was explained during the validation period.

Although Linear Regression provides measurable improvement over the naive
baseline, overall performance remains modest. The substantially larger RMSE
relative to MAE suggests that the model makes some large prediction errors,
likely during high-demand periods. Further residual analysis is therefore
needed before evaluating regularized alternatives.

In [ ]:
linear_residuals = y_val - y_val_pred_linear

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    y_val_pred_linear,
    linear_residuals,
    alpha=0.5
)

plt.axhline(
    y=0,
    linestyle="--"
)

plt.xlabel("Predicted Appliance Consumption (Wh)")
plt.ylabel("Residual (Wh)")
plt.title("Linear Regression Residuals vs Predicted Values")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))

plt.scatter(
    y_val,
    y_val_pred_linear,
    alpha=0.5
)

plt.xlabel("Actual Appliance Consumption (Wh)")
plt.ylabel("Predicted Appliance Consumption (Wh)")
plt.title("Actual vs Predicted Appliance Consumption")

plt.tight_layout()
plt.show()

In [ ]:
error_analysis = pd.DataFrame({
    "Actual": y_val,
    "Predicted": y_val_pred_linear,
    "Residual": linear_residuals
})

error_analysis["Absolute_Error"] = (
    error_analysis["Residual"].abs()
)

error_analysis.sort_values(
    "Absolute_Error",
    ascending=False
).head(20)

In [ ]:
high_demand_mask = y_val >= 200

normal_mae = mean_absolute_error(
    y_val[~high_demand_mask],
    y_val_pred_linear[~high_demand_mask]
)

high_demand_mae = mean_absolute_error(
    y_val[high_demand_mask],
    y_val_pred_linear[high_demand_mask]
)

print(f"MAE below 200 Wh: {normal_mae:.2f} Wh")
print(f"MAE at or above 200 Wh: {high_demand_mae:.2f} Wh")

In [ ]:
print(
    "High-demand observations:",
    high_demand_mask.sum()
)

print(
    "High-demand percentage:",
    round(high_demand_mask.mean() * 100, 2),
    "%"
)

### Linear Regression Error Analysis

Residual analysis revealed that Linear Regression does not perform uniformly
across consumption levels. The largest errors were associated with
high-demand observations, where the model substantially underpredicted
actual appliance consumption.

For example, an observed consumption value of 840 Wh was predicted at
approximately 105 Wh, resulting in an error of approximately 735 Wh.
Several other high-demand observations produced errors greater than 500 Wh.

The residual-versus-predicted plot also exhibited systematic structure rather
than a completely random distribution around zero, suggesting that important
patterns in appliance demand are not adequately captured by the linear
specification.

This behavior helps explain the difference between MAE and RMSE. Because
RMSE penalizes large errors more heavily, severe underprediction during
high-demand periods disproportionately increases RMSE.

These results motivate further investigation of regularized linear models
for coefficient stability and, later, nonlinear models for potentially
capturing more complex relationships.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

In [ ]:
ridge_pipeline.fit(
    X_train,
    y_train
)

In [ ]:
y_val_pred_ridge = ridge_pipeline.predict(X_val)

In [ ]:
ridge_mae = mean_absolute_error(
    y_val,
    y_val_pred_ridge
)

ridge_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        y_val_pred_ridge
    )
)

ridge_r2 = r2_score(
    y_val,
    y_val_pred_ridge
)

print(f"Ridge MAE:  {ridge_mae:.2f} Wh")
print(f"Ridge RMSE: {ridge_rmse:.2f} Wh")
print(f"Ridge R²:   {ridge_r2:.3f}")

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Naive Baseline",
        "Linear Regression",
        "Ridge (alpha=1)"
    ],

    "MAE": [
        baseline_mae,
        linear_mae,
        ridge_mae
    ],

    "RMSE": [
        baseline_rmse,
        linear_rmse,
        ridge_rmse
    ],

    "R2": [
        baseline_r2,
        linear_r2,
        ridge_r2
    ]
})

comparison

In [ ]:
import numpy as np

ridge_alphas = np.logspace(
    -4,
    4,
    50
)

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

In [ ]:
ridge_grid = {
    "model__alpha": ridge_alphas
}

In [ ]:
ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_grid,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    n_jobs=-1
)

In [ ]:
ridge_search.fit(
    X_train,
    y_train
)

In [ ]:
print(
    "Best Ridge alpha:",
    ridge_search.best_params_["model__alpha"]
)

In [ ]:
print(
    "Best CV RMSE:",
    -ridge_search.best_score_
)

In [ ]:
best_ridge = ridge_search.best_estimator_

In [ ]:
y_val_pred_ridge_tuned = best_ridge.predict(
    X_val
)

In [ ]:
ridge_tuned_mae = mean_absolute_error(
    y_val,
    y_val_pred_ridge_tuned
)

ridge_tuned_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        y_val_pred_ridge_tuned
    )
)

ridge_tuned_r2 = r2_score(
    y_val,
    y_val_pred_ridge_tuned
)

print(f"Tuned Ridge MAE:  {ridge_tuned_mae:.2f} Wh")
print(f"Tuned Ridge RMSE: {ridge_tuned_rmse:.2f} Wh")
print(f"Tuned Ridge R²:   {ridge_tuned_r2:.3f}")

In [ ]:
ridge_coefficients = pd.Series(
    best_ridge.named_steps["model"].coef_,
    index=X_train.columns
)

In [ ]:
ridge_coefficients.sort_values(
    key=abs,
    ascending=False
)

In [ ]:
linear_coefficients = pd.Series(
    linear_pipeline.named_steps["model"].coef_,
    index=X_train.columns
)

In [ ]:
coefficient_comparison = pd.DataFrame({
    "Linear Regression": linear_coefficients,
    "Ridge": ridge_coefficients
})

coefficient_comparison["Linear_abs"] = (
    coefficient_comparison["Linear Regression"].abs()
)

coefficient_comparison["Ridge_abs"] = (
    coefficient_comparison["Ridge"].abs()
)

In [ ]:
coefficient_comparison.sort_values(
    "Linear_abs",
    ascending=False
).head(20)

In [ ]:
coefficient_comparison.loc[
    [
        "rv1",
        "rv2",
        "T6",
        "T_out",
        "T7",
        "T9"
    ]
]

### Ridge Regression

Ridge Regression was evaluated because exploratory analysis revealed
substantial multicollinearity among the environmental sensor variables.
The regularization parameter was tuned using time-aware cross-validation
within the training period.

The optimal Ridge penalty was approximately 1526.42. The tuned model
achieved an MAE of 47.27 Wh, RMSE of 85.07 Wh, and R² of 0.149 on the
chronologically held-out validation period.

Compared with ordinary Linear Regression, Ridge reduced MAE from 51.20 Wh
to 47.27 Wh and RMSE from 86.12 Wh to 85.07 Wh while increasing R² from
0.128 to 0.149. The larger improvement in MAE than RMSE suggests that
regularization improved typical prediction errors but did not fully resolve
the large errors associated with high-demand periods.

Overall, Ridge provided a modest improvement over ordinary Linear Regression,
supporting the value of regularization in this highly correlated feature
space. However, the remaining error indicates that coefficient shrinkage
alone is insufficient to capture the full structure of appliance energy
demand.

In [ ]:
from sklearn.linear_model import Lasso

lasso_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(max_iter=20000))
])

In [ ]:
lasso_alphas = np.logspace(
    -4,
    2,
    60
)

In [ ]:
lasso_grid = {
    "model__alpha": lasso_alphas
}

In [ ]:
lasso_search = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid=lasso_grid,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    n_jobs=-1
)

In [ ]:
lasso_search.fit(
    X_train,
    y_train
)

In [ ]:
print(
    "Best Lasso alpha:",
    lasso_search.best_params_["model__alpha"]
)

print(
    "Best CV RMSE:",
    -lasso_search.best_score_
)

In [ ]:
best_lasso = lasso_search.best_estimator_

y_val_pred_lasso = best_lasso.predict(X_val)

In [ ]:
lasso_mae = mean_absolute_error(
    y_val,
    y_val_pred_lasso
)

lasso_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        y_val_pred_lasso
    )
)

lasso_r2 = r2_score(
    y_val,
    y_val_pred_lasso
)

print(f"Lasso MAE:  {lasso_mae:.2f} Wh")
print(f"Lasso RMSE: {lasso_rmse:.2f} Wh")
print(f"Lasso R²:   {lasso_r2:.3f}")

In [ ]:
lasso_coefficients = pd.Series(
    best_lasso.named_steps["model"].coef_,
    index=X_train.columns
)

In [ ]:
n_zero = (lasso_coefficients == 0).sum()
n_nonzero = (lasso_coefficients != 0).sum()

print("Zero coefficients:", n_zero)
print("Selected features:", n_nonzero)
print("Total features:", len(lasso_coefficients))

In [ ]:
lasso_coefficients[
    lasso_coefficients == 0
].index.tolist()

In [ ]:
lasso_coefficients[
    lasso_coefficients != 0
].sort_values(
    key=abs,
    ascending=False
)

In [ ]:
lasso_coefficients[
    ["rv1", "rv2"]
]

In [ ]:
removed_features = lasso_coefficients[
    lasso_coefficients == 0
].index.tolist()

print("Features removed by Lasso:")

for feature in removed_features:
    print(feature)

In [ ]:
selected_features = (
    lasso_coefficients[
        lasso_coefficients != 0
    ]
    .sort_values(
        key=abs,
        ascending=False
    )
)

selected_features

### Lasso Regression

Lasso Regression was evaluated to determine whether L1 regularization could
reduce model complexity through automatic feature selection. The
regularization parameter was tuned using time-aware cross-validation, which
selected an alpha of approximately 2.36.

The tuned Lasso model achieved an MAE of 48.14 Wh, RMSE of 85.84 Wh, and
R² of 0.134 on the validation period. Although its predictive performance
was slightly weaker than Ridge Regression, Lasso produced a substantially
sparser model.

Of the 33 candidate predictors, 18 coefficients were reduced exactly to
zero, leaving 15 selected features. Notably, both deliberately random
variables (rv1 and rv2) received zero coefficients, demonstrating Lasso's
ability to suppress predictors that provide little incremental predictive
value.

These results illustrate a tradeoff between predictive performance and model
sparsity: Ridge achieved the strongest validation performance among the
linear models evaluated so far, whereas Lasso achieved substantial feature
reduction with only a modest loss in predictive accuracy.

In [ ]:
from sklearn.linear_model import ElasticNet

elastic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet(max_iter=50000))
])

In [ ]:
elastic_alphas = np.logspace(
    -3,
    2,
    30
)

elastic_l1_ratios = [
    0.1,
    0.25,
    0.5,
    0.75,
    0.9,
    0.95
]

In [ ]:
elastic_grid = {
    "model__alpha": elastic_alphas,
    "model__l1_ratio": elastic_l1_ratios
}

In [ ]:
elastic_search = GridSearchCV(
    estimator=elastic_pipeline,
    param_grid=elastic_grid,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    n_jobs=-1
)

In [ ]:
elastic_search.fit(
    X_train,
    y_train
)

In [ ]:
print(
    "Best Elastic Net alpha:",
    elastic_search.best_params_["model__alpha"]
)

print(
    "Best Elastic Net l1_ratio:",
    elastic_search.best_params_["model__l1_ratio"]
)

print(
    "Best CV RMSE:",
    -elastic_search.best_score_
)

In [ ]:
best_elastic = elastic_search.best_estimator_

y_val_pred_elastic = best_elastic.predict(
    X_val
)

In [ ]:
elastic_mae = mean_absolute_error(
    y_val,
    y_val_pred_elastic
)

elastic_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        y_val_pred_elastic
    )
)

elastic_r2 = r2_score(
    y_val,
    y_val_pred_elastic
)

print(f"Elastic Net MAE:  {elastic_mae:.2f} Wh")
print(f"Elastic Net RMSE: {elastic_rmse:.2f} Wh")
print(f"Elastic Net R²:   {elastic_r2:.3f}")

In [ ]:
elastic_coefficients = pd.Series(
    best_elastic.named_steps["model"].coef_,
    index=X_train.columns
)

In [ ]:
elastic_zero = (
    elastic_coefficients == 0
).sum()

elastic_nonzero = (
    elastic_coefficients != 0
).sum()

print("Zero coefficients:", elastic_zero)
print("Selected features:", elastic_nonzero)
print("Total features:", len(elastic_coefficients))

In [ ]:
elastic_coefficients[
    elastic_coefficients == 0
].index.tolist()

In [ ]:
elastic_coefficients[
    ["rv1", "rv2"]
]

In [ ]:
linear_model_results = pd.DataFrame({
    "Model": [
        "Naive Baseline",
        "Linear Regression",
        "Ridge",
        "Lasso",
        "Elastic Net"
    ],

    "MAE": [
        baseline_mae,
        linear_mae,
        ridge_tuned_mae,
        lasso_mae,
        elastic_mae
    ],

    "RMSE": [
        baseline_rmse,
        linear_rmse,
        ridge_tuned_rmse,
        lasso_rmse,
        elastic_rmse
    ],

    "R2": [
        baseline_r2,
        linear_r2,
        ridge_tuned_r2,
        lasso_r2,
        elastic_r2
    ]
})

linear_model_results.sort_values("RMSE")

In [ ]:
plot_results = (
    linear_model_results
    .sort_values("RMSE")
)

plt.figure(figsize=(10, 6))

plt.bar(
    plot_results["Model"],
    plot_results["RMSE"]
)

plt.ylabel("Validation RMSE (Wh)")
plt.xlabel("Model")
plt.title("Validation RMSE Across Linear Models")

plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

## Linear Model Comparison

Four linear modeling approaches were evaluated against a naive mean
baseline: ordinary Linear Regression, Ridge Regression, Lasso Regression,
and Elastic Net.

All linear models improved upon the naive baseline. Ridge Regression achieved
the strongest validation performance, with an MAE of 47.27 Wh, RMSE of
85.07 Wh, and R² of 0.149.

Regularization was particularly relevant because exploratory analysis
identified substantial multicollinearity among the environmental sensor
variables. Ridge substantially reduced the magnitude of several unstable
coefficients while retaining all predictors.

Lasso provided the strongest feature-selection effect, reducing 18 of 33
coefficients to zero and retaining only 15 predictors. Elastic Net produced
an intermediate solution, retaining 23 predictors. Both Lasso and Elastic
Net eliminated the deliberately random variables rv1 and rv2.

Although regularization improved coefficient behavior and produced modest
gains in predictive performance, residual analysis showed that the linear
models continued to substantially underpredict high-consumption events.
This suggests that regularization alone cannot address all of the structure
in appliance energy demand and motivates evaluation of nonlinear models.

## Decision Tree Regression

The linear and regularized regression models provided useful baselines but showed limited predictive performance, particularly for high-energy-consumption observations. This suggests that nonlinear relationships and interactions among the predictors may be important.

A Decision Tree Regressor is therefore evaluated as the first nonlinear model. Unlike linear regression, a decision tree recursively partitions the feature space and can capture nonlinear relationships and feature interactions without requiring them to be specified explicitly.

In [ ]:
print(X_train.shape)
print(y_train.shape)

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(
    random_state=42
)

dt.fit(X_train, y_train)

In [ ]:
dt_train_pred = dt.predict(X_train)
dt_val_pred = dt.predict(X_val)

In [ ]:
dt_train_mae = mean_absolute_error(y_train, dt_train_pred)
dt_train_rmse = mean_squared_error(y_train, dt_train_pred) ** 0.5
dt_train_r2 = r2_score(y_train, dt_train_pred)

dt_val_mae = mean_absolute_error(y_val, dt_val_pred)
dt_val_rmse = mean_squared_error(y_val, dt_val_pred) ** 0.5
dt_val_r2 = r2_score(y_val, dt_val_pred)

print("Decision Tree - Training")
print(f"MAE:  {dt_train_mae:.2f}")
print(f"RMSE: {dt_train_rmse:.2f}")
print(f"R²:   {dt_train_r2:.3f}")

print("\nDecision Tree - Validation")
print(f"MAE:  {dt_val_mae:.2f}")
print(f"RMSE: {dt_val_rmse:.2f}")
print(f"R²:   {dt_val_r2:.3f}")

In [ ]:
param_grid_dt = {
    "max_depth": [3, 5, 8, 12, 16, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10, 20]
}

In [ ]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

dt_tuned = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=param_grid_dt,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    n_jobs=-1
)

dt_tuned.fit(X_train, y_train)

In [ ]:
print("Best parameters:")
print(dt_tuned.best_params_)

print("\nBest CV RMSE:")
print(f"{-dt_tuned.best_score_:.2f}")

In [ ]:
best_dt = dt_tuned.best_estimator_

dt_tuned_train_pred = best_dt.predict(X_train)
dt_tuned_val_pred = best_dt.predict(X_val)

print("Tuned Decision Tree - Training")
print(f"MAE:  {mean_absolute_error(y_train, dt_tuned_train_pred):.2f}")
print(f"RMSE: {mean_squared_error(y_train, dt_tuned_train_pred) ** 0.5:.2f}")
print(f"R²:   {r2_score(y_train, dt_tuned_train_pred):.3f}")

print("\nTuned Decision Tree - Validation")
print(f"MAE:  {mean_absolute_error(y_val, dt_tuned_val_pred):.2f}")
print(f"RMSE: {mean_squared_error(y_val, dt_tuned_val_pred) ** 0.5:.2f}")
print(f"R²:   {r2_score(y_val, dt_tuned_val_pred):.3f}")

### Decision Tree Interpretation

Time-series cross-validation selected a relatively shallow Decision Tree (`max_depth=3`, `min_samples_leaf=2`, `min_samples_split=2`), with a cross-validation RMSE of 102.59.

The tuned model achieved a training RMSE of 95.97 (R² = 0.193) and a validation RMSE of 89.04 (R² = 0.068). The relatively modest gap between training and validation performance indicates that constraining tree depth reduced severe overfitting. However, the low training and validation R² values suggest that the shallow tree has relatively high bias and is unable to capture much of the variation in appliance energy consumption.

The tuned Decision Tree also did not outperform the strongest regularized linear model on the validation data. This motivates evaluating Random Forest, which combines multiple diversified Decision Trees to capture nonlinear relationships while reducing the variance associated with individual trees.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

In [ ]:
rf_train_pred = rf.predict(X_train)
rf_val_pred = rf.predict(X_val)

print("Random Forest - Training")
print(f"MAE:  {mean_absolute_error(y_train, rf_train_pred):.2f}")
print(f"RMSE: {mean_squared_error(y_train, rf_train_pred) ** 0.5:.2f}")
print(f"R²:   {r2_score(y_train, rf_train_pred):.3f}")

print("\nRandom Forest - Validation")
print(f"MAE:  {mean_absolute_error(y_val, rf_val_pred):.2f}")
print(f"RMSE: {mean_squared_error(y_val, rf_val_pred) ** 0.5:.2f}")
print(f"R²:   {r2_score(y_val, rf_val_pred):.3f}")

### Random Forest Baseline Interpretation

The baseline Random Forest achieved very strong training performance (RMSE = 24.41, R² = 0.948) but substantially worse validation performance (RMSE = 105.72, R² = -0.314).

The large training-validation performance gap indicates severe overfitting: the model captured the training observations extremely well but failed to generalize to the later validation period. The negative validation R² further indicates that the model performed poorly on unseen future observations.

Although Random Forest reduces the variance of individual Decision Trees through averaging, unrestricted trees can still produce an overly complex ensemble. Therefore, the next step is to regularize the forest by tuning tree depth, minimum leaf size, feature sampling, and related complexity controls using time-series cross-validation.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist_rf = {
    "n_estimators": [200, 300, 500],
    "max_depth": [3, 5, 8, 12, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10, 20],
    "max_features": ["sqrt", 0.5, 1.0]
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ),
    param_distributions=param_dist_rf,
    n_iter=30,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)

In [ ]:
print("Best parameters:")
print(rf_search.best_params_)

print("\nBest CV RMSE:")
print(f"{-rf_search.best_score_:.2f}")

In [ ]:
best_rf = rf_search.best_estimator_

rf_tuned_train_pred = best_rf.predict(X_train)
rf_tuned_val_pred = best_rf.predict(X_val)

print("Tuned Random Forest - Training")
print(f"MAE:  {mean_absolute_error(y_train, rf_tuned_train_pred):.2f}")
print(f"RMSE: {mean_squared_error(y_train, rf_tuned_train_pred) ** 0.5:.2f}")
print(f"R²:   {r2_score(y_train, rf_tuned_train_pred):.3f}")

print("\nTuned Random Forest - Validation")
print(f"MAE:  {mean_absolute_error(y_val, rf_tuned_val_pred):.2f}")
print(f"RMSE: {mean_squared_error(y_val, rf_tuned_val_pred) ** 0.5:.2f}")
print(f"R²:   {r2_score(y_val, rf_tuned_val_pred):.3f}")

### Tuned Random Forest Interpretation

Hyperparameter tuning substantially improved the Random Forest's ability to generalize. The baseline forest severely overfit the training data (training R² = 0.948; validation R² = -0.314), whereas the tuned model produced much more comparable training and validation performance.

The tuned Random Forest achieved a training RMSE of 88.38 (R² = 0.316) and a validation RMSE of 86.20 (R² = 0.127). Although training performance decreased substantially after regularization, validation performance improved considerably, demonstrating the importance of controlling model complexity.

The tuned Random Forest was competitive with, but did not outperform, the strongest regularized linear model on the validation period. This motivates evaluating a boosting approach, where trees are constructed sequentially to improve the errors remaining from the current ensemble.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(
    random_state=42
)

gbr.fit(X_train, y_train)

gbr_train_pred = gbr.predict(X_train)
gbr_val_pred = gbr.predict(X_val)

print("Gradient Boosting - Training")
print(f"MAE:  {mean_absolute_error(y_train, gbr_train_pred):.2f}")
print(f"RMSE: {mean_squared_error(y_train, gbr_train_pred) ** 0.5:.2f}")
print(f"R²:   {r2_score(y_train, gbr_train_pred):.3f}")

print("\nGradient Boosting - Validation")
print(f"MAE:  {mean_absolute_error(y_val, gbr_val_pred):.2f}")
print(f"RMSE: {mean_squared_error(y_val, gbr_val_pred) ** 0.5:.2f}")
print(f"R²:   {r2_score(y_val, gbr_val_pred):.3f}")

In [ ]:
param_dist_gbr = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4, 5],
    "min_samples_leaf": [5, 10, 20, 30],
    "subsample": [0.6, 0.8, 1.0]
}

gbr_search = RandomizedSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_distributions=param_dist_gbr,
    n_iter=30,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

gbr_search.fit(X_train, y_train)

In [ ]:
print("Best parameters:")
print(gbr_search.best_params_)

print("\nBest CV RMSE:")
print(f"{-gbr_search.best_score_:.2f}")

In [ ]:
best_gbr = gbr_search.best_estimator_

gbr_tuned_train_pred = best_gbr.predict(X_train)
gbr_tuned_val_pred = best_gbr.predict(X_val)

print("Tuned Gradient Boosting - Training")
print(f"MAE:  {mean_absolute_error(y_train, gbr_tuned_train_pred):.2f}")
print(f"RMSE: {mean_squared_error(y_train, gbr_tuned_train_pred) ** 0.5:.2f}")
print(f"R²:   {r2_score(y_train, gbr_tuned_train_pred):.3f}")

print("\nTuned Gradient Boosting - Validation")
print(f"MAE:  {mean_absolute_error(y_val, gbr_tuned_val_pred):.2f}")
print(f"RMSE: {mean_squared_error(y_val, gbr_tuned_val_pred) ** 0.5:.2f}")
print(f"R²:   {r2_score(y_val, gbr_tuned_val_pred):.3f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.scatter(
    y_val,
    gbr_tuned_val_pred,
    alpha=0.3,
    label="Gradient Boosting"
)

plt.plot(
    [y_val.min(), y_val.max()],
    [y_val.min(), y_val.max()],
    "r--",
    label="Perfect prediction"
)

plt.xlabel("Actual Energy Consumption")
plt.ylabel("Predicted Energy Consumption")
plt.title("Gradient Boosting: Actual vs Predicted")
plt.legend()
plt.show()

### Actual vs. Predicted Values

The actual-versus-predicted plot shows that the tuned Gradient Boosting model performs more effectively within the common range of appliance energy consumption but systematically underpredicts high-consumption observations.

As actual consumption increases, predictions remain concentrated within a substantially narrower range. This indicates that the model has difficulty capturing extreme energy-demand events. These large underprediction errors also contribute disproportionately to RMSE.

Therefore, although Gradient Boosting achieved the strongest validation performance among the evaluated models, important predictive limitations remain, particularly during periods of unusually high appliance energy consumption.

In [ ]:
import pandas as pd

error_analysis = pd.DataFrame({
    "actual": y_val,
    "predicted": gbr_tuned_val_pred
})

error_analysis["residual"] = (
    error_analysis["actual"] - error_analysis["predicted"]
)

error_analysis["absolute_error"] = (
    error_analysis["residual"].abs()
)

error_analysis.sort_values(
    "absolute_error",
    ascending=False
).head(10)

### Analysis of Largest Prediction Errors

Inspection of the ten largest absolute errors confirms the pattern observed in the actual-versus-predicted plot. The model's largest errors occur during high-consumption observations, with actual appliance energy consumption ranging approximately from 670 to 870 Wh.

All ten cases have large positive residuals, indicating substantial underprediction. For example, an actual consumption value of 840 Wh was predicted as approximately 103 Wh, producing an error of about 737 Wh.

These results indicate that the model performs substantially worse during extreme energy-consumption events than during typical consumption periods. The relatively small number of high-consumption observations and the strongly right-skewed target distribution may contribute to this behavior.

In [ ]:
error_analysis["consumption_group"] = pd.cut(
    error_analysis["actual"],
    bins=[0, 100, 200, 400, float("inf")],
    labels=["0-100", "101-200", "201-400", ">400"]
)

error_by_group = (
    error_analysis
    .groupby("consumption_group", observed=True)
    .agg(
        observations=("actual", "size"),
        MAE=("absolute_error", "mean"),
        mean_residual=("residual", "mean")
    )
)

error_by_group

### Error Across Consumption Levels

Prediction performance varied substantially across consumption levels. For observations between 0 and 100 Wh, the model had an MAE of 33.52 Wh and tended to overpredict consumption. Performance was strongest between 101 and 200 Wh, where MAE was 28.44 Wh and the mean residual was close to zero.

However, prediction error increased sharply for high-consumption observations. For values between 201 and 400 Wh, MAE increased to 158.55 Wh, with a mean residual of +158.46 Wh. For observations above 400 Wh, MAE reached 418.55 Wh, and the mean residual was +418.55 Wh.

This confirms that the model systematically underpredicts high-consumption events. Therefore, the overall validation metrics conceal substantial variation in predictive performance across different consumption levels.

In [ ]:
plt.figure(figsize=(12, 5))

plt.scatter(
    range(len(error_analysis)),
    error_analysis["residual"],
    alpha=0.4
)

plt.axhline(
    y=0,
    linestyle="--"
)

plt.xlabel("Validation Observation")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Gradient Boosting Residuals Across Validation Period")

plt.show()

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": best_gbr.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
)

feature_importance.head(15)

In [ ]:
from sklearn.inspection import permutation_importance

perm_importance = permutation_importance(
    best_gbr,
    X_val,
    y_val,
    scoring="neg_root_mean_squared_error",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    "feature": X_val.columns,
    "importance": perm_importance.importances_mean,
    "std": perm_importance.importances_std
})

perm_df = perm_df.sort_values(
    "importance",
    ascending=False
)

perm_df.head(15)

In [ ]:
perm_df.head(15)

In [ ]:
top_perm = perm_df.head(15).sort_values(
    "importance",
    ascending=True
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_perm["feature"],
    top_perm["importance"]
)

plt.xlabel("Permutation Importance")
plt.ylabel("Feature")
plt.title("Top 15 Features - Permutation Importance")

plt.tight_layout()
plt.show()

### Permutation Importance

Permutation importance indicated that the cyclical time-of-day features (`hour_cos` and `hour_sin`) were the most influential predictors of validation performance, followed by room temperature (`T3`).

The substantial importance of the hour features suggests that appliance energy consumption follows meaningful daily usage patterns. `lights`, `RH_6`, and `Tdewpoint` provided smaller contributions, while many other temperature and humidity variables had relatively little individual permutation importance.

These results describe the features the trained model relied upon for prediction and should not be interpreted as evidence of causal relationships. Additionally, correlated predictors may share or substitute predictive information, which can reduce their individual permutation importance.

### Final Model Selection

Tuned Gradient Boosting was selected for final evaluation. It achieved the lowest validation RMSE (84.87) and highest validation R² (0.153) among the evaluated models.

Ridge Regression produced a slightly better cross-validation RMSE (95.66 versus 97.36 for Gradient Boosting), indicating that the performance difference between the strongest models was small. Therefore, Gradient Boosting was selected based on its held-out validation performance rather than assuming that its greater complexity guarantees superior performance.

The final model will now be evaluated once on the previously untouched test set.

In [ ]:
gbr_test_pred = best_gbr.predict(X_test)

test_mae = mean_absolute_error(y_test, gbr_test_pred)
test_rmse = mean_squared_error(y_test, gbr_test_pred) ** 0.5
test_r2 = r2_score(y_test, gbr_test_pred)

print("Final Gradient Boosting - Test Set")
print(f"MAE:  {test_mae:.2f}")
print(f"RMSE: {test_rmse:.2f}")
print(f"R²:   {test_r2:.3f}")

### Final Test Performance

The selected Gradient Boosting model was evaluated once on the previously untouched test set. It achieved an MAE of 63.49 Wh, RMSE of 90.42 Wh, and R² of 0.010.

Performance was weaker than on the validation set, where the model achieved an MAE of 48.07 Wh, RMSE of 84.87 Wh, and R² of 0.153. The near-zero test R² indicates that the model explained very little of the variation in appliance energy consumption during the final unseen period.

Combined with the earlier error analysis, these results demonstrate that predicting appliance energy consumption across future time periods remains challenging, particularly for relatively rare high-consumption events.

The test set was used only for final evaluation, and no additional model tuning was performed after observing these results.

In [ ]:
test_errors = pd.DataFrame({
    "actual": y_test,
    "predicted": gbr_test_pred
})

test_errors["residual"] = (
    test_errors["actual"] - test_errors["predicted"]
)

test_errors["absolute_error"] = (
    test_errors["residual"].abs()
)

test_errors.sort_values(
    "absolute_error",
    ascending=False
).head(10)

### Final Test Error Analysis

The final test-set error analysis confirmed the pattern previously observed on the validation data. The largest prediction errors were consistently associated with high appliance-energy-consumption observations.

For example, an actual consumption value of 850 Wh was predicted as approximately 172 Wh, resulting in an underprediction of approximately 678 Wh. Similar large underpredictions occurred throughout the ten largest absolute test errors.

The persistence of this pattern across both validation and test periods indicates that the model has difficulty capturing relatively rare high-demand events. Although the model performs more effectively within the common consumption range, its predictions tend to remain compressed toward moderate values during extreme consumption periods.

## Conclusion

This project evaluated multiple regression approaches for forecasting household appliance energy consumption using a chronological train-validation-test framework.

Linear Regression was used as an initial benchmark, followed by Ridge, Lasso, and Elastic Net regularization. Nonlinear approaches including Decision Tree, Random Forest, and Gradient Boosting were subsequently evaluated and tuned using time-series cross-validation.

Hyperparameter tuning substantially reduced overfitting in the tree-based models. The tuned Gradient Boosting model achieved the strongest held-out validation performance, with an RMSE of 84.87 Wh and R² of 0.153, and was therefore selected for final evaluation.

On the previously untouched test set, the model achieved:

- MAE: 63.49 Wh
- RMSE: 90.42 Wh
- R²: 0.010

The deterioration from validation to test performance demonstrates the difficulty of generalizing appliance-energy predictions across future time periods.

Error analysis revealed an important limitation: the model consistently underpredicted high-consumption events. This behavior appeared in both the validation and test sets, indicating that relatively rare energy-demand spikes were not adequately captured by the model.

Permutation importance showed that cyclical time-of-day variables were among the most influential predictors, highlighting the importance of daily usage patterns in appliance energy consumption.

Overall, the project demonstrates that model complexity alone does not guarantee better generalization. Careful chronological validation, hyperparameter regularization, error analysis, and transparent reporting of model limitations are essential when developing predictive models for time-dependent data.